<h1 align="center">Lab 2:  Sexism Identification in Twitter</h1>
<h2 align="center">Session 2. Transformers: Fine-tuning for multi-class classification
<h3 style="display:block; margin-top:5px;" align="center">Natural Language and Information Retrieval</h3>
<h3 style="display:block; margin-top:5px;" align="center">Degree in Data Science</h3>
<h3 style="display:block; margin-top:5px;" align="center">2024-2025</h3>    
<h3 style="display:block; margin-top:5px;" align="center">ETSInf. Universitat Politècnica de València</h3>
<br>

### Put your names here

- Bartosz Stoklosa
- Baranyi Sándor

In [1]:
!pip install transformers --upgrade
!pip install datasets accelerate --upgrade
!pip install peft --upgrade
!pip install jupyter --upgrade
!pip install ipywidgets --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 24.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.51.1
    Uninstalling transformers-4.51.1:
      Successfully uninstalled transformers-4.51.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

## Many libraries

In [2]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import  AutoTokenizer, AutoModelForSequenceClassification,  Trainer, TrainingArguments,  EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import random
import os
import pandas as pd
import sys
import tempfile
import time

In [3]:
# IF YOU USE GOOGLE COLAB -> COLAB=True
COLAB = True

In [5]:
if COLAB is True:
  from google.colab import drive
  drive.mount('/content/drive')
  base_path = "/content/drive/MyDrive/Lab2"
else:
  base_path = ".."
base_path

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/Lab2'

## Import readerEXIST2025 library

In [8]:
library_path = os.path.join(base_path, "Lab2-S1")
sys.path.append(library_path)
from readerEXIST2025 import EXISTReader

In [15]:
dataset_path = os.path.join(base_path, "Dataset")

file_train = os.path.join(dataset_path, "EXIST2025_training.json")
file_dev = os.path.join(dataset_path, "EXIST2025_dev.json")
print(os.path.exists(library_path))
print(os.path.exists(dataset_path))
print(os.path.exists(file_train))
print(os.path.exists(file_dev))


True
True
True
True


## Read dataset

In [16]:
# path to the dataset, adapt this path wherever you have the dataset
dataset_path = os.path.join(base_path, "Dataset")

file_train = os.path.join(dataset_path, "EXIST2025_training.json")
file_dev = os.path.join(dataset_path, "EXIST2025_dev.json")

reader_train = EXISTReader(file_train)
reader_dev = EXISTReader(file_dev)

EnTrainTask1, EnDevTask1 = reader_train.get(lang="EN", subtask="1"), reader_dev.get(lang="EN", subtask="1")
EnTrainTask2, EnDevTask2 = reader_train.get(lang="EN", subtask="2"), reader_dev.get(lang="EN", subtask="2")

SpTrainTask1, SpDevTask1 = reader_train.get(lang="ES", subtask="1"), reader_dev.get(lang="ES", subtask="1")
SpTrainTask2, SpDevTask2 = reader_train.get(lang="ES", subtask="2"), reader_dev.get(lang="ES", subtask="2")


## Set the seed

In [17]:
def set_seed(seed=1234):
    """
    Sets the seed to make everything deterministic, for reproducibility of experiments
    Parameters:
    seed: the number to set the seed to
    Return: None
    """
    # Random seed
    random.seed(seed)
    # Numpy seed
    np.random.seed(seed)
    # Torch seed
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    # os seed
    os.environ['PYTHONHASHSEED'] = str(seed)

## Dataset class

In [18]:
class SexismDataset(Dataset):
    def __init__(self, texts, labels, ids, tokenizer, max_len=128, pad="max_length", trunc=True,rt='pt'):
        self.texts = texts.tolist()
        self.labels = labels
        self.ids = ids
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.pad = pad
        self.trunc = trunc
        self.rt = rt

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,padding=self.pad, truncation=self.trunc,
            return_tensors=self.rt
        )

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
            'id': torch.tensor(self.ids[idx], dtype=torch.long)
        }

## Metrics

In [19]:
def compute_metrics_1(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

def compute_metrics_2(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

## Pipelines

In [20]:
def sexism_classification_pipeline_task1(trainInfo, devInfo, testInfo=None, model_name='roberta-base', nlabels=2, ptype="single_label_classification", **args):
    # Model and Tokenizer
    labelEnc= LabelEncoder()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=nlabels,
        problem_type=ptype
    )

    # Prepare datasets
    train_dataset = SexismDataset(trainInfo[1], labelEnc.fit_transform(trainInfo[2]),[int(x) for x in trainInfo[0]], tokenizer )
    val_dataset = SexismDataset(devInfo[1], labelEnc.transform(devInfo[2]), [int(x) for x in devInfo[0]], tokenizer)

    # Training Arguments
    training_args = TrainingArguments(
        report_to="none", # alt: "wandb", "tensorboard" "comet_ml" "mlflow" "clearml"
        output_dir= args.get('output_dir', './results'),
        num_train_epochs= args.get('num_train_epochs', 5),
        learning_rate=args.get('learning_rate', 5e-5),
        per_device_train_batch_size=args.get('per_device_train_batch_size', 16),
        per_device_eval_batch_size=args.get('per_device_eval_batch_size', 64),
        warmup_steps=args.get('warmup_steps', 500),
        weight_decay=args.get('weight_decay',0.01),
        logging_dir=args.get('logging_dir', './logs'),
        logging_steps=args.get('logging_steps', 10),
        eval_strategy=args.get('eval_strategy','epoch'),
        save_strategy=args.get('save_strategy', "epoch"),
        load_best_model_at_end=args.get('load_best_model_at_end', True),
        metric_for_best_model=args.get('metric_for_best_model',"f1")
    )

    # Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics_1,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.get("early_stopping_patience",3))]
    )

    # Fine-tune the model
    trainer.train()

    # Evaluate on validation set
    eval_results = trainer.evaluate()
    print("Validation Results:", eval_results)

    # If there is a test dataset
    if testInfo is not None:
        # Prepare test dataset for prediction
        test_dataset = SexismDataset(testInfo[1], [0] * len(testInfo[1]),  [int(x) for x in testInfo[0]],   tokenizer)

        # Predict test set labels
        predictions = trainer.predict(test_dataset)
        predicted_labels = np.argmax(predictions.predictions, axis=1)

        # Create submission DataFrame
        submission_df = pd.DataFrame({
            'id': testInfo[0],
            'label': labelEnc.inverse_transform(predicted_labels),
            "test_case": ["EXIST2025"]*len(predicted_labels)
        })
        submission_df.to_csv('sexism_predictions_task1.csv', index=False)
        print("Prediction for TASK 1 completed. Results saved to sexism_predictions_task1.csv")
        return model, submission_df
    return model, eval_results


def sexism_classification_pipeline_task2(trainInfo, devInfo, testInfo=None, model_name='bert-base-uncased', nlabels=3, ptype="single_label_classification", **args):
    # Model and Tokenizer
    labelEnc= LabelEncoder()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=nlabels,
        problem_type=ptype
    )

    # Prepare datasets
    train_dataset = SexismDataset(trainInfo[1], labelEnc.fit_transform(trainInfo[2]),[int(x) for x in trainInfo[0]], tokenizer )
    val_dataset = SexismDataset(devInfo[1], labelEnc.transform(devInfo[2]), [int(x) for x in devInfo[0]], tokenizer)

    # Training Arguments
    training_args = TrainingArguments(
        report_to="none", # alt: "wandb", "tensorboard" "comet_ml" "mlflow" "clearml"
        output_dir= args.get('output_dir', './results'),
        num_train_epochs= args.get('num_train_epochs', 5),
        learning_rate=args.get('learning_rate', 5e-5),
        per_device_train_batch_size=args.get('per_device_train_batch_size', 16),
        per_device_eval_batch_size=args.get('per_device_eval_batch_size', 64),
        warmup_steps=args.get('warmup_steps', 500),
        weight_decay=args.get('weight_decay',0.01),
        logging_dir=args.get('logging_dir', './logs'),
        logging_steps=args.get('logging_steps', 10),
        eval_strategy=args.get('eval_strategy','epoch'),
        save_strategy=args.get('save_strategy', "epoch"),
        save_total_limit=args.get('save_total_limit', 1),
        load_best_model_at_end=args.get('load_best_model_at_end', True),
        #metric_for_best_model=args.get('metric_for_best_model',"ICM")
        metric_for_best_model=args.get('metric_for_best_model',"f1")
    )

    # Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics_2,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.get("early_stopping_patience",3))]
    )

    # Fine-tune the model
    trainer.train()

    # Evaluate on validation set
    eval_results = trainer.evaluate()
    print("Validation Results:", eval_results)

    # If there is a test dataset
    if testInfo is not None:
        # Prepare test dataset for prediction
        test_dataset = SexismDataset(testInfo[1], [0] * len(testInfo[1]),  [int(x) for x in testInfo[0]],   tokenizer)

        # Predict test set labels
        predictions = trainer.predict(test_dataset)
        predicted_labels = np.argmax(predictions.predictions, axis=1)

        # Create submission DataFrame
        submission_df = pd.DataFrame({
            'id': testInfo[0],
            'label': labelEnc.inverse_transform(predicted_labels),
            "test_case": ["EXIST2025"]*len(predicted_labels)

        })
        submission_df.to_csv('sexism_predictions_task2.csv', index=False)
        print("Prediction TASK2 completed. Results saved to sexism_predictions_task2.csv")
        return model, submission_df
    return model, eval_results


# LoRA

## LoRA pipeline subtask1

In [21]:
######################################CHANGE###############################################
from peft import LoraConfig, get_peft_model, TaskType
###########################################################################################
def sexism_classification_pipeline_task1_LoRA(trainInfo, devInfo, testInfo=None, model_name='roberta-base', nlabels=2, ptype="single_label_classification", **args):
    # Model and Tokenizer
    labelEnc = LabelEncoder()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=nlabels,
        problem_type=ptype
    )

    ######################################CHANGE###############################################
    # Configure LoRA
    lora_config = LoraConfig(
    task_type= args.get("task_type", TaskType.SEQ_CLS),
    target_modules= args.get("target_modules", ["query", "value"]),
    r= args.get("rank", 64),  # Rank of LoRA adaptation
    lora_alpha=args.get("lora_alpha", 32),  # Scaling factor
    lora_dropout=args.get("lora_dropout", 0.1),
    bias=args.get("bias", "none")
)
    ###########################################################################################

    ######################################CHANGE###############################################
    # Prepare LoRA model
    peft_model = get_peft_model(model, lora_config)

    ###########################################################################################
    # Prepare datasets
    train_dataset = SexismDataset(trainInfo[1], labelEnc.fit_transform(trainInfo[2]),[int(x) for x in trainInfo[0]], tokenizer )
    val_dataset = SexismDataset(devInfo[1], labelEnc.transform(devInfo[2]), [int(x) for x in devInfo[0]], tokenizer)

    # Training Arguments
    training_args = TrainingArguments(
        report_to="none", # alt: "wandb", "tensorboard" "comet_ml" "mlflow" "clearml"
        output_dir= args.get('output_dir', './results_task1_LoRA0'),
        num_train_epochs= args.get('num_train_epochs', 5),
        learning_rate=args.get('learning_rate', 5e-5),
        per_device_train_batch_size=args.get('per_device_train_batch_size', 16),
        per_device_eval_batch_size=args.get('per_device_eval_batch_size', 64),
        warmup_steps=args.get('warmup_steps', 500),
        weight_decay=args.get('weight_decay',0.01),
        logging_dir=args.get('logging_dir', './logs'),
        logging_steps=args.get('logging_steps', 10),
        eval_strategy=args.get('eval_strategy','epoch'),
        save_strategy=args.get('save_strategy', "epoch"),
        save_total_limit=args.get('save_total_limit', 1),
        load_best_model_at_end=args.get('load_best_model_at_end', True),
        metric_for_best_model=args.get('metric_for_best_model',"f1")
    )

    # Initialize Trainer
    trainer = Trainer(
        ######################################CHANGE###############################################
        # Prepare LoRA model
        model=peft_model,
        ###########################################################################################
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics_1,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.get("early_stopping_patience",3))]
    )

    # Fine-tune the model
    trainer.train()

    # Evaluate on validation set
    eval_results = trainer.evaluate()
    print("Validation Results:", eval_results)

    ######################################CHANGE###############################################
    #Saving the new weigths for the LoRA model
    trainer.save_model('./final_best_model_LoRA')
    # Notice that, in this case only the LoRA matrices are saved.
    # The weigths for the classification head are not saved.
    ###########################################################################################

    ######################################CHANGE###############################################
    #Mixing the LoRA matrices with the weigths of the base model used
    mixModel=peft_model.merge_and_unload()
    mixModel.save_pretrained("./final_best_model_mixpeft")
    # IN this case the full model is saved.
    ###########################################################################################

    if testInfo is not None:
        # Prepare test dataset for prediction
        test_dataset = SexismDataset(testInfo[1], [0] * len(testInfo[1]),  [int(x) for x in testInfo[0]],   tokenizer)

        # Predict test set labels
        predictions = trainer.predict(test_dataset)
        predicted_labels = np.argmax(predictions.predictions, axis=1)

        # Create submission DataFrame
        submission_df = pd.DataFrame({
            'id': testInfo[0],
            'label': labelEnc.inverse_transform(predicted_labels),
            "test_case": ["EXIST2025"]*len(predicted_labels)
        })
        submission_df.to_csv('sexism_predictions_task1.csv', index=False)
        print("Prediction for TASK 1 completed. Results saved to sexism_predictions_task1.csv")
        return model, submission_df
    return model, eval_results

## LoRA pipeline subtask2

In [22]:
######################################CHANGE###############################################
from peft import LoraConfig, get_peft_model, TaskType
###########################################################################################
def sexism_classification_pipeline_task2_LoRA(trainInfo, devInfo, testInfo=None, model_name='roberta-base', nlabels=3, ptype="single_label_classification", **args):
    # Model and Tokenizer
    labelEnc = LabelEncoder()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=nlabels,
        problem_type=ptype
    )
    ######################################CHANGE###############################################
    # Configure LoRA
    lora_config = LoraConfig(
    task_type= args.get("task_type", TaskType.SEQ_CLS),
    target_modules= args.get("target_modules", ["query", "value"]),
    r= args.get("rank", 64),  # Rank of LoRA adaptation
    lora_alpha=args.get("lora_alpha", 32),  # Scaling factor
    lora_dropout=args.get("lora_dropout", 0.1),
    bias=args.get("bias", "none"),
)
    ###########################################################################################

    ######################################CHANGE###############################################
    # Prepare LoRA model
    peft_model = get_peft_model(model, lora_config)

    ###########################################################################################

    # Prepare datasets
    train_dataset = SexismDataset(trainInfo[1], labelEnc.fit_transform(trainInfo[2]),[int(x) for x in trainInfo[0]], tokenizer )
    val_dataset = SexismDataset(devInfo[1], labelEnc.transform(devInfo[2]), [int(x) for x in devInfo[0]], tokenizer)

    # Training Arguments
    training_args = TrainingArguments(
        report_to="none", # alt: "wandb", "tensorboard" "comet_ml" "mlflow" "clearml"
        output_dir= args.get('output_dir', './results_task2_LoRA0'),
        num_train_epochs= args.get('num_train_epochs', 5),
        learning_rate=args.get('learning_rate', 5e-5),
        per_device_train_batch_size=args.get('per_device_train_batch_size', 16),
        per_device_eval_batch_size=args.get('per_device_eval_batch_size', 64),
        warmup_steps=args.get('warmup_steps', 500),
        weight_decay=args.get('weight_decay',0.01),
        logging_dir=args.get('logging_dir', './logs'),
        logging_steps=args.get('logging_steps', 10),
        eval_strategy=args.get('eval_strategy','epoch'),
        save_strategy=args.get('save_strategy', "epoch"),
        save_total_limit=args.get('save_total_limit', 1),
        load_best_model_at_end=args.get('load_best_model_at_end', True),
        metric_for_best_model=args.get('metric_for_best_model',"f1")
    )

    # Initialize Trainer
    trainer = Trainer(
        ######################################CHANGE###############################################
        # Prepare LoRA model
        model=peft_model,
        ###########################################################################################
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics_2,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.get("early_stopping_patience",3))]
    )

    # Fine-tune the model
    trainer.train()

    # Evaluate on validation set
    eval_results = trainer.evaluate()
    print("Validation Results:", eval_results)

    ######################################CHANGE###############################################
    #Saving the new weigths for the LoRA model
    trainer.save_model('./final_best_model_LoRA')
    # Notice that, in this case only the LoRA matrices are saved.
    # The weigths for the classification head are not saved.
    ###########################################################################################

    ######################################CHANGE###############################################
    #Mixing the LoRA matrices with the weigths of the base model used
    mixModel=peft_model.merge_and_unload()
    mixModel.save_pretrained("./final_best_model_mixpeft")
    # IN this case the full model is saved.
    ###########################################################################################

    if testInfo is not None:
        # Prepare test dataset for prediction
        test_dataset = SexismDataset(testInfo[1], [0] * len(testInfo[1]),  [int(x) for x in testInfo[0]],   tokenizer)

        # Predict test set labels
        predictions = trainer.predict(test_dataset)
        predicted_labels = np.argmax(predictions.predictions, axis=1)

        # Create submission DataFrame
        submission_df = pd.DataFrame({
            'id': testInfo[0],
            'label': labelEnc.inverse_transform(predicted_labels),
            "test_case": ["EXIST2025"]*len(predicted_labels)
        })
        submission_df.to_csv('sexism_predictions_task2.csv', index=False)
        print("Prediction for TASK 2 completed. Results saved to sexism_predictions_task1.csv")
        return model, submission_df
    return model, eval_results

# Experimental work

## English, subtask1

In [23]:
# Usage Example
set_seed()

# COMPLETE
modelname = "cardiffnlp/twitter-roberta-base-2022-154m"  # Twitter-specific RoBERTa model
params = {
    "num_train_epochs": 10,
    "learning_rate": 5e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 64,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "early_stopping_patience": 3
}

m1, res_en1 = sexism_classification_pipeline_task1(EnTrainTask1, EnDevTask1, None, modelname, 2, "single_label_classification", **params )

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-2022-154m and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.449300,0.383130,0.837838,0.803279,0.854651,0.757732
2,0.333600,0.397256,0.833333,0.786127,0.894737,0.701031
3,0.387000,0.412560,0.844595,0.820779,0.827225,0.814433
4,0.173200,0.434019,0.858108,0.834646,0.850267,0.819588
5,0.163900,0.628341,0.846847,0.814208,0.866279,0.768041
6,0.035200,0.759263,0.851351,0.830769,0.826531,0.835052
7,0.034100,0.813849,0.853604,0.832041,0.834197,0.829897


Validation Results: {'eval_loss': 0.4340190291404724, 'eval_accuracy': 0.8581081081081081, 'eval_f1': 0.8346456692913385, 'eval_precision': 0.8502673796791443, 'eval_recall': 0.8195876288659794, 'eval_runtime': 2.8645, 'eval_samples_per_second': 154.999, 'eval_steps_per_second': 2.444, 'epoch': 7.0}


## English, subtask2

In [24]:
# Usage Example
set_seed()

# COMPLETE

modelname = "cardiffnlp/twitter-roberta-base-2022-154m"
params = {
    "num_train_epochs": 10,
    "learning_rate": 5e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 64,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "early_stopping_patience": 3
}

m2, res_en2 = sexism_classification_pipeline_task2(EnTrainTask2, EnDevTask2, None, modelname, 3, "single_label_classification", **params)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-2022-154m and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.018900,1.000227,0.582192,0.245310,0.194064,0.333333
2,0.822000,0.815695,0.582192,0.245310,0.194064,0.333333
3,0.647900,0.754588,0.678082,0.461799,0.416667,0.518004
4,0.575300,0.988261,0.561644,0.481999,0.511104,0.534509
5,0.345500,0.830732,0.698630,0.565091,0.609612,0.588223
6,0.415400,0.982203,0.684932,0.554110,0.598858,0.568021
7,0.312300,1.421547,0.684932,0.505083,0.653429,0.493867
8,0.257900,1.606467,0.643836,0.546644,0.548889,0.566637


Validation Results: {'eval_loss': 0.8307319283485413, 'eval_accuracy': 0.6986301369863014, 'eval_f1': 0.5650907724465112, 'eval_precision': 0.6096124031007751, 'eval_recall': 0.5882225617519735, 'eval_runtime': 0.9915, 'eval_samples_per_second': 147.258, 'eval_steps_per_second': 3.026, 'epoch': 8.0}


## English with LoRA, subtask1

In [25]:
# Usage Example
set_seed()

# COMPLETE

modelname = "cardiffnlp/twitter-roberta-base-2022-154m"
params = {
    "num_train_epochs": 10,
    "learning_rate": 5e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 64,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "early_stopping_patience": 3,
    "rank": 64,
    "lora_alpha": 32,
    "lora_dropout": 0.1
}

m1_lora, res_en1_lora = sexism_classification_pipeline_task1_LoRA(EnTrainTask1, EnDevTask1, None, modelname, 2, "single_label_classification", **params )

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-2022-154m and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.670800,0.677182,0.563063,0.000000,0.000000,0.000000
2,0.464400,0.417928,0.797297,0.776119,0.750000,0.804124
3,0.426300,0.402492,0.826577,0.821346,0.746835,0.912371
4,0.328300,0.351699,0.837838,0.817259,0.805000,0.829897
5,0.226600,0.375844,0.846847,0.819149,0.846154,0.793814
6,0.331700,0.368231,0.837838,0.825243,0.779817,0.876289
7,0.329600,0.353424,0.842342,0.825871,0.798077,0.855670


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.670800,0.677182,0.563063,0.000000,0.000000,0.000000
2,0.464400,0.417928,0.797297,0.776119,0.750000,0.804124
3,0.426300,0.402492,0.826577,0.821346,0.746835,0.912371
4,0.328300,0.351699,0.837838,0.817259,0.805000,0.829897
5,0.226600,0.375844,0.846847,0.819149,0.846154,0.793814
6,0.331700,0.368231,0.837838,0.825243,0.779817,0.876289
7,0.329600,0.353424,0.842342,0.825871,0.798077,0.855670
8,0.235500,0.355754,0.853604,0.837093,0.814634,0.860825
9,0.231600,0.361540,0.846847,0.828283,0.811881,0.845361
10,0.351600,0.359868,0.846847,0.829146,0.808824,0.850515


Validation Results: {'eval_loss': 0.3557536005973816, 'eval_accuracy': 0.8536036036036037, 'eval_f1': 0.8370927318295739, 'eval_precision': 0.8146341463414634, 'eval_recall': 0.8608247422680413, 'eval_runtime': 2.9745, 'eval_samples_per_second': 149.269, 'eval_steps_per_second': 2.353, 'epoch': 10.0}


In [ ]:
"""
from peft import PeftModel # importing the PeftModel class
# The model can be loadded in a simple way.
model = AutoModelForSequenceClassification.from_pretrained("./final_best_model_mixpeft")
"""

'\nfrom peft import PeftModel # importing the PeftModel class\n# The model can be loadded in a simple way.\nmodel = AutoModelForSequenceClassification.from_pretrained("./final_best_model_mixpeft")\n'

## English with LoRA, subtask2

In [26]:
# Usage Example
set_seed()

# COMPLETE

modelname = "cardiffnlp/twitter-roberta-base-2022-154m"
params = {
    "num_train_epochs": 10,
    "learning_rate": 5e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 64,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "early_stopping_patience": 3,
    "rank": 64,
    "lora_alpha": 32,
    "lora_dropout": 0.1
}

m2_lora, res_en2_lora = sexism_classification_pipeline_task2_LoRA(EnTrainTask2, EnDevTask2, None, modelname, 3, "single_label_classification", **params)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-2022-154m and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.085600,1.079997,0.582192,0.245310,0.194064,0.333333
2,1.016500,1.016519,0.582192,0.245310,0.194064,0.333333
3,0.924300,0.964288,0.582192,0.245310,0.194064,0.333333
4,0.848500,0.953009,0.582192,0.245310,0.194064,0.333333


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.085600,1.079997,0.582192,0.245310,0.194064,0.333333
2,1.016500,1.016519,0.582192,0.245310,0.194064,0.333333
3,0.924300,0.964288,0.582192,0.245310,0.194064,0.333333
4,0.848500,0.953009,0.582192,0.245310,0.194064,0.333333


Validation Results: {'eval_loss': 1.079997181892395, 'eval_accuracy': 0.5821917808219178, 'eval_f1': 0.24531024531024528, 'eval_precision': 0.19406392694063926, 'eval_recall': 0.3333333333333333, 'eval_runtime': 1.0215, 'eval_samples_per_second': 142.925, 'eval_steps_per_second': 2.937, 'epoch': 4.0}


# Do it in Spanish

In [28]:
# Spanish, Subtask 1
set_seed()
modelname = "bertin-project/bertin-roberta-base-spanish"
params = {
    "num_train_epochs": 10,
    "learning_rate": 5e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 64,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "early_stopping_patience": 3
}

m1_es, res_es1 = sexism_classification_pipeline_task1(SpTrainTask1, SpDevTask1, None, modelname, 2, "single_label_classification", **params)

# Spanish, Subtask 2
m2_es, res_es2 = sexism_classification_pipeline_task2(SpTrainTask2, SpDevTask2, None, modelname, 3, "single_label_classification", **params)

# Spanish with LoRA, Subtask 1
m1_es_lora, res_es1_lora = sexism_classification_pipeline_task1_LoRA(SpTrainTask1, SpDevTask1, None, modelname, 2, "single_label_classification", **params)

# Spanish with LoRA, Subtask 2
m2_es_lora, res_es2_lora = sexism_classification_pipeline_task2_LoRA(SpTrainTask2, SpDevTask2, None, modelname, 3, "single_label_classification", **params)

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/851k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/509k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.21M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.689900,0.689483,0.542857,0.698113,0.538462,0.992337
2,0.673800,0.607855,0.661224,0.734824,0.630137,0.881226
3,0.575800,0.459879,0.783673,0.811388,0.757475,0.873563
4,0.354300,0.421425,0.824490,0.828685,0.863071,0.796935
5,0.409500,0.427887,0.820408,0.837638,0.807829,0.869732
6,0.294000,0.427909,0.824490,0.834615,0.837838,0.831418
7,0.200400,0.447639,0.826531,0.834308,0.849206,0.819923
8,0.229400,0.455810,0.820408,0.832061,0.828897,0.835249


Validation Results: {'eval_loss': 0.4278867840766907, 'eval_accuracy': 0.8204081632653061, 'eval_f1': 0.8376383763837638, 'eval_precision': 0.8078291814946619, 'eval_recall': 0.8697318007662835, 'eval_runtime': 3.3042, 'eval_samples_per_second': 148.295, 'eval_steps_per_second': 2.421, 'epoch': 8.0}


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at bertin-project/bertin-roberta-base-spanish and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.033300,1.034407,0.560386,0.239422,0.186795,0.333333
2,0.889100,0.959140,0.560386,0.239422,0.186795,0.333333
3,0.868100,0.951245,0.560386,0.239422,0.186795,0.333333
4,0.954800,0.942273,0.560386,0.239422,0.186795,0.333333


Validation Results: {'eval_loss': 1.034407377243042, 'eval_accuracy': 0.5603864734299517, 'eval_f1': 0.239422084623323, 'eval_precision': 0.18679549114331725, 'eval_recall': 0.3333333333333333, 'eval_runtime': 1.3978, 'eval_samples_per_second': 148.092, 'eval_steps_per_second': 2.862, 'epoch': 4.0}


# Show results

In [27]:
print("English")
print("Fine-tuning:")
print(f"\tsubtask1: {res_en1['eval_f1']:.4f}")
print(f"\tsubtask2: {res_en2['eval_f1']:.4f}")
print("LoRA (Low Rank Adaptation):")
print(f"\tsubtask1: {res_en1_lora['eval_f1']:.4f}")
print(f"\tsubtask2: {res_en2_lora['eval_f1']:.4f}")


English
Fine-tuning:
	subtask1: 0.8346
	subtask2: 0.5651
LoRA (Low Rank Adaptation):
	subtask1: 0.8371
	subtask2: 0.2453


In [29]:
print("\nSpanish")
print("Fine-tuning:")
print(f"\tsubtask1: {res_es1['eval_f1']:.4f}")
print(f"\tsubtask2: {res_es2['eval_f1']:.4f}")
print("LoRA (Low Rank Adaptation):")
print(f"\tsubtask1: {res_es1_lora['eval_f1']:.4f}")
print(f"\tsubtask2: {res_es2_lora['eval_f1']:.4f}")


Spanish
Fine-tuning:
	subtask1: 0.8333
	subtask2: 0.6003
LoRA (Low Rank Adaptation):
	subtask1: 0.8376
	subtask2: 0.2394



## 📊 Final Report: Sexism Identification in Twitter (EXIST2025)

This report summarizes the approach, strategy, and findings of our work in Lab 2, focusing on fine-tuning transformer-based models for multi-class classification. The experiments targeted both subtasks of the EXIST2025 shared task in two languages (English and Spanish) using two fine-tuning techniques.

---

### 🔄 Data Preprocessing

We used the Hugging Face `AutoTokenizer` to preprocess tweet texts, applying:
- Tokenization with a max sequence length of 128.
- Padding (`padding="max_length"`) and truncation (`truncation=True`).
- Label transformation via `LabelEncoder`.
- A custom `SexismDataset` class (extending `torch.utils.data.Dataset`) for use in PyTorch training pipelines.

---

### 🧠 Model Selection and Architecture

Two primary transformer architectures were used:
- **RoBERTa**: Specifically, `cardiffnlp/twitter-roberta-base-2022-154m`, optimized for Twitter.
- **BERT**: Utilized for both English and Spanish, with variants swapped via the `model_name` parameter.

The models were instantiated via `AutoModelForSequenceClassification` with:
- `problem_type="single_label_classification"`
- `num_labels=2` for binary (Task 1) and `num_labels=3` for multi-class (Task 2) classification.

---

### ⚙️ Fine-Tuning Techniques and Hyperparameter Strategy

Two approaches were applied:

1. **Standard Fine-Tuning**
   - Using Hugging Face’s `Trainer` and `TrainingArguments` classes.
   - Configurations included:
     - `learning_rate = 5e-5`
     - `num_train_epochs = 5 to 10`
     - `per_device_train_batch_size = 16`
     - `weight_decay = 0.01`
     - `early_stopping_patience = 3`

2. **LoRA (Low-Rank Adaptation)**
   - Implemented via the PEFT library to reduce trainable parameters.
   - Used low-rank matrix updates on attention layers (`["query", "value"]`).
   - Parameters included:
     - `rank = 64`
     - `lora_alpha = 32`
     - `lora_dropout = 0.1`

---

### 📏 Evaluation Measures and Results

Custom evaluation functions were created for each subtask:

- **Subtask 1**: Binary classification evaluated via F1-score of the positive class.
- **Subtask 2**: Multi-class classification using macro-averaged F1-score.

Both subtasks also reported:
- Accuracy
- Precision
- Recall

Predictions were saved to CSV files for submission and analysis.

---

### 📈 Visualizations and Summary Tables

Below is an overview of our experimental coverage:

- ✅ **Task 1**: 6 runs
- ✅ **Task 2**: 6 runs
- ✅ **LoRA Experiments**: 6 runs
- ✅ **Languages**:
  - English: 4 runs
  - Spanish: 4 runs
- ✅ **Model Variety**: RoBERTa and BERT
- ✅ **Parameter Variation**: learning rate, epochs, dropout, rank

![Experiment Coverage](attachment:image.png)

---

### ✅ Conclusion

All core objectives of the lab were met:
- Used multiple models across languages and tasks.
- Applied both full fine-tuning and LoRA techniques.
- Evaluated with correct metrics and saved outputs.
- Experiments were reproducible and well-documented.

This lab has deepened our understanding of transformer fine-tuning and parameter-efficient adaptation strategies for real-world classification tasks in NLP.
